In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 8.1 Learning as Least Squares: Ridge, the SVD Spectrum, and Generalization

In [ ]:
from ecp.style import header, use_style

use_style()
header(
    volume="Chapter VIII — The Linear Algebra of Learning",
    number="8.1",
    title="Learning as Least Squares: Ridge, the SVD Spectrum, and "
    "Generalization",
    blurb="Strip a learning problem to its chassis and Chapter II is "
    "driving: fitting is least squares, regularization is a filter on the "
    "singular spectrum, generalization is a bias-variance ledger that "
    "closes to the percent — and at the interpolation threshold the test "
    "error spikes and descends again, all of it visible in one SVD.",
    difficulty="advanced",
    estimate="105–135 min",
)

## Notebook overview

Chapter VIII reads machine learning as this course's material wearing a
new vocabulary, and it starts where the translation is exact: a linear
model on synthetic data whose truth is known. Ordinary least squares
and ridge regression are [§2.3](../02-orthogonality/least-squares-four-ways.ipynb)
and [§2.4](../02-orthogonality/pseudoinverse-regularization.ipynb); what
is new is the *statistical* reading, and every claim of it is gated.
The ridge solution's **SVD filter form** — shrink each singular
direction by $\sigma_i^2/(\sigma_i^2+\lambda)$ — matches the matrix
solve at $10^{-15}$; its $\lambda \to 0$ limit is the pseudoinverse
solution; the **effective degrees of freedom** $\sum\sigma_i^2/
(\sigma_i^2+\lambda)$ runs from the rank to zero as the dial turns; and
the solution norm decreases monotonically along the whole path.

The centrepiece is the **bias–variance decomposition**, done as an
honest Monte Carlo: three thousand fresh noise draws on a dedicated
stream, predictions averaged, and the ledger
$\text{bias}^2 + \text{variance} + \sigma^2$ gated against test error
*measured on independently noisy targets* — agreement within 3%
(measured: under 1%), with the cross-terms vanishing only in
expectation, which is what makes the gate a statement about statistics
rather than an algebraic identity checked against itself. Then the
modern coda {cite}`belkin2019`: sweeping polynomial features across
the interpolation threshold produces the **double-descent** curve —
test error spiking by fifteen orders of magnitude at $p = n$ (the
spike's *size* is the machine's smallest singular value talking;
only its existence is gated) and descending again in the
overparameterized regime, where the **minimum-norm interpolator**
$A^{+}y$ fits the training data to $10^{-14}$ and is shortest among
all interpolators by an exact Pythagorean identity. The honest ending:
on this problem the second descent never beats the well-chosen small
model — double descent is a phenomenon, not a free lunch, and the
measurement says so.

> **How to read a check.** A `validate` line prints ✓ or ✗ by comparing a
> result against something the computation did not assume. A ✗ flags a
> mismatch to investigate, never a verdict on its own.

> **Scope.** Hastie, Tibshirani and Friedman {cite}`hastie2009`
> Chapters 3 and 7; Belkin et al. {cite}`belkin2019` for double
> descent; Strang {cite}`strang2019learning` for the linear-algebra
> framing. The SVD machinery is Chapter IV's; the statistical layer is
> what this chapter adds.

## Theory in brief

### Ridge as a spectral filter

With the thin SVD $A = U\Sigma V^{\top}$ of the $n \times p$ design,

```{math}
:label: eq-ll-filter
w_\lambda \;=\; (A^{\top}A + \lambda I)^{-1}A^{\top}y
\;=\; \sum_i \frac{\sigma_i}{\sigma_i^2 + \lambda}\,
(u_i^{\top}y)\,v_i ,
```

each direction kept by the **filter factor**
$\phi_i = \sigma_i^2/(\sigma_i^2+\lambda) \in (0, 1)$: directions with
$\sigma_i^2 \gg \lambda$ pass, those with $\sigma_i^2 \ll \lambda$ are
shrunk away — [§2.4](../02-orthogonality/pseudoinverse-regularization.ipynb)'s
truncation, made smooth. Two consequences are theorems along the whole
path: the **effective degrees of freedom**
$\mathrm{df}(\lambda) = \sum_i \phi_i$ falls monotonically from
$\operatorname{rank}A$ to $0$, and $\lVert w_\lambda\rVert$ falls
monotonically too (every $\phi_i$ does).

### The generalization ledger

For fresh data $y^* = f + \varepsilon^*$, the expected test error of
any fixed-design linear fit splits exactly:

```{math}
:label: eq-ll-biasvar
\mathbb{E}\,\lVert \hat f - y^* \rVert^2/n
\;=\; \underbrace{\lVert \mathbb{E}\hat f - f\rVert^2/n}_{\text{bias}^2}
\;+\; \underbrace{\mathbb{E}\lVert \hat f - \mathbb{E}\hat f
\rVert^2/n}_{\text{variance}}
\;+\; \sigma^2 ,
```

with the cross-terms zero *in expectation* — so a Monte Carlo over
noise draws must close the ledger only up to sampling error, and
gating that closure is gating the statistics, not an identity.
Regularization trades the two terms: $\lambda$ up, variance down,
bias up.

### The interpolation threshold

With $p \ge n$ the training data can be fit exactly, and among all
interpolators the pseudoinverse choice $w = A^{+}y$ is the shortest —
$\lVert w + z\rVert^2 = \lVert w\rVert^2 + \lVert z\rVert^2$ for any
null-space $z$, exactly ([§2.4](../02-orthogonality/pseudoinverse-regularization.ipynb)'s
Pythagoras). Near $p = n$ the design's smallest singular value
collapses and the variance term detonates — the double-descent spike
{cite}`belkin2019` — while past it the minimum-norm choice regularizes
implicitly and the error descends again.

---
## Setup

Data only: the ground truth, the noisy draw, and the Legendre design with
its SVD. The filter form — the object every claim below reads through —
is Exercise 1's build.

The Setup below holds this notebook's data and instruments — nothing you
are asked to build. It is collapsed so the building stays yours; expand it
whenever you want the details.

<!-- setup-policy: v2 -->

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.polynomial import legendre

from ecp import validate
from ecp.style import use_style

use_style()
rng = np.random.default_rng(0)  # every random array below comes from this seed

EPS = np.finfo(float).eps


# data: the planted ground truth.
def f_true(x):
    """The ground truth every error below is measured against."""
    return np.sin(1.25 * np.pi * x + 0.3) + 0.3 * x


N_TR, N_TE = 40, 400
SIGMA = 0.3
P_FEAT = 15
x_tr = np.sort(rng.uniform(-1, 1, N_TR))
x_te = np.linspace(-1, 1, N_TE)
y_tr = f_true(x_tr) + SIGMA * rng.standard_normal(N_TR)
f_te = f_true(x_te)

A_tr = legendre.legvander(x_tr, P_FEAT - 1)
A_te = legendre.legvander(x_te, P_FEAT - 1)
U_d, s_d, Vt_d = np.linalg.svd(A_tr, full_matrices=False)
print(f"design: {N_TR} samples x {P_FEAT} Legendre features, "
      f"cond = {s_d[0]/s_d[-1]:.1f}")



## Exercise 1: Ridge is a filter on the spectrum

**Part a)** Write `ridge_svd(y, lam)` — the filter form of
{eq}`eq-ll-filter`: apply $U^{\top}$, scale coefficient $i$ by
$\sigma_i/(\sigma_i^2+\lambda)$, map back through $V$ — and gate it
against the
matrix solve $(A^{\top}A + \lambda I)^{-1}A^{\top}y$ at three
$\lambda$, to $10^{-12}$ — one identity, two routes, and the filter
route is the one every claim below reads from.

**Write this one yourself** — every claim in this notebook reads
through the filter you write here.

**Part b)** Gate the $\lambda \to 0$ limit: at $\lambda = 10^{-10}$
the ridge solution matches the pseudoinverse solution
`np.linalg.pinv(A) @ y` to $10^{-9}$ — the filter factors all reach 1
and [§2.4](../02-orthogonality/pseudoinverse-regularization.ipynb)
reappears.

**Part c)** Draw the filter factors $\phi_i$ against $\sigma_i$ for
$\lambda \in \{10^{-3}, 10^{-1}, 10\}$: a smooth gate sliding across
the spectrum, keeping strong directions and shrinking weak ones — the
picture of what "regularization strength" means.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.below(
    worst_form, 1e-12,
    "the SVD filter form equals the matrix solve (Eq. 1)",
    "sigma/(sigma^2 + lambda) direction by direction against the normal-"
    "equations route — 2.3 and 2.4, reconciled at rounding level",
)
validate.below(
    gap_limit, 1e-9,
    "and lambda -> 0 recovers the pseudoinverse solution",
    "all filter factors reach 1: ridge contains least squares as its "
    "zero-regularization face",
)

## Exercise 2: Degrees of freedom, and the monotone path

**Part a)** Gate the limits of
$\mathrm{df}(\lambda) = \sum_i \sigma_i^2/(\sigma_i^2+\lambda)$: at
$\lambda = 10^{-10}$ it equals the rank (15) to $10^{-6}$; at
$\lambda = 10^{6}$ it sits below $10^{-3}$ — the dial runs the model
from "all fifteen directions" to "none of them".

**Part b)** Gate the two path monotonicities over thirty $\lambda$
spanning eight decades: $\mathrm{df}(\lambda)$ strictly decreasing and
$\lVert w_\lambda\rVert$ strictly decreasing — both are sums of
individually decreasing filter terms, so the gates are theorems
checked, not observations hoped for.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.check(
    abs(dof_lo - P_FEAT) < 1e-6 and dof_hi < 1e-3,
    "effective degrees of freedom run from the rank to zero",
    f"{dof_lo:.4f} at lambda = 1e-10 and {dof_hi:.0e} at 1e6 — the dial's "
    "endpoints, priced in directions",
)
validate.check(
    mono_dof and mono_norm,
    "and both df and the solution norm fall strictly along the path",
    "every filter factor is individually decreasing in lambda, so both "
    "sums are — theorems checked over eight decades",
)

## Exercise 3: The bias–variance ledger, closed by Monte Carlo

{eq}`eq-ll-biasvar` splits in expectation; this exercise makes the
expectation empirical and gates the closure.

**Part a)** On a **dedicated noise stream** — the course's Monte Carlo
rule — draw $R = 3000$ fresh training-noise vectors, refit ridge at
$\lambda \in \{10^{-4}, 0.1, 10\}$, and record all predictions on the
test grid.

**Part b)** Assemble the ledger per $\lambda$: bias² from the mean
prediction against the noiseless truth, variance from the spread, plus
$\sigma^2$. Measure the test error *independently* — against noisy
targets $f + \varepsilon^*$ with fresh $\varepsilon^*$ per replicate —
and gate the closure to the manifest's 3% (measured: under 1%). The
cross-terms cancel only in expectation, so this gate certifies the
statistics, not an identity recomputed.

**Part c)** Gate the trade: variance strictly decreasing across the
three $\lambda$, bias² strictly increasing — the dial working as
{eq}`eq-ll-biasvar` says it must. Draw the three ledgers as stacked
bars beside the fitted curves at the same $\lambda$.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.check(
    closure_worst < 0.03,
    "bias^2 + variance + noise equals the measured test error within 3% "
    "(Eq. 2)",
    f"worst closure {closure_worst:.2%} over three lambdas and 3000 "
    "dedicated-stream replicates — the cross-terms cancel in expectation "
    "and the Monte Carlo says they did in sample",
)
validate.check(
    trade_ok,
    "with variance falling and bias rising as the dial turns",
    "the trade Eq. 2 promises, measured as two strictly monotone "
    "three-term sequences",
)

## Exercise 4: Across the interpolation threshold

Now hold the data fixed ($n = 20$ samples) and sweep the *model*:
Legendre features $p = 1, \dots, 40$, each fit by the minimum-norm
rule $w = A^{+}y$.

**Part a)** Gate the interpolation regime's entry ticket: at
$p = 40 \ge n$, training residual below $10^{-10}$ — the model passes
through every training point.

**Part b)** The sweep: test error against the truth for every $p$.
Gate the double-descent *shape* as orderings (its magnitudes belong to
the machine): the spike near $p = n$ exceeds $10^{3}\times$ both the
best underparameterized error and the $p = 40$ error — the smallest
singular value's detonation, gated only as *existing* — and the
$p = 40$ error sits under $2.0$ (descended), while the spike's actual
height ($\sim10^{16}$ here) is *reported*.

**Part c)** The honest ending, stated and gated: the second descent's
floor ($1.0$) never reaches the well-chosen small model's ($0.01$) on
this problem — gate `err(40) > 10 x err(best p < 12)`. Double descent
is a real phenomenon and not a free lunch, and the same figure shows
both.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.check(
    res_interp < 1e-10,
    "past the threshold the model interpolates: training error at "
    "rounding level",
    f"{res_interp:.0e} at p = 40 — the entry ticket to the "
    "overparameterized regime",
)
validate.check(
    spike > 1e3 * max(under_best, over_err) and over_err < 2.0,
    "the double-descent shape holds as orderings",
    f"spike {spike:.0e} over 1000x both flanks; p = 40 descended to "
    f"{over_err:.2f} — the spike's height is the smallest singular "
    "value's business (reported), the shape is the mathematics'",
)
validate.check(
    over_err > 10.0 * under_best,
    "and the second descent never catches the well-chosen small model",
    f"{over_err:.2f} against {under_best:.3f}: a phenomenon, not a free "
    "lunch — gated so the notebook cannot oversell its own coda",
)

## Exercise 5: The minimum-norm interpolator, exactly shortest

**Part a)** At $p = 40$, characterize the interpolators: any
$w = A^{+}y + z$ with $z \in \operatorname{null}(A)$ fits the training
data exactly. Gate the null space's dimension: $40 - 20 = 20$
(`matrix_rank` over an $O(1)$ gap).

**Part b)** Gate minimality *exactly*: for ten random null-space
perturbations, $\lVert A^{+}y + z\rVert^2 = \lVert A^{+}y\rVert^2 +
\lVert z\rVert^2$ to $10^{-10}$ relative (Pythagoras — the
pseudoinverse solution is orthogonal to the null space), so every
other interpolator is strictly longer. This is
[§2.4](../02-orthogonality/pseudoinverse-regularization.ipynb)'s
identity carrying the *implicit regularization* story: gradient
descent from zero converges to exactly this shortest interpolator,
which is why the overparameterized regime generalizes at all —
and why [§8.6](low-rank-lora-quantization.ipynb) will care about
norms of weight updates.

In [ ]:
# (solution hidden on the public site)


```{admonition} With your assistant
:class: tip
Gradient descent was asserted, not run. Ask your assistant for
`gd_least_squares(A, y, steps, lr)` iterating $w \leftarrow w -
\eta A^{\top}(Aw - y)$ from $w_0 = 0$ on the $p = 40$ design, then
check it against the mathematics rather than a demo: (i) it converges
to the minimum-norm interpolator to $10^{-8}$ (never to any other —
the iterates stay in the row space, which is the whole mechanism);
(ii) its training error decreases monotonically for
$\eta < 2/\sigma_1^2$ and diverges for $\eta$ above it (verify both
sides of the edge); (iii) early stopping at increasing step counts
traces solution norms that increase monotonically toward
$\lVert A^{+}y\rVert$ — iteration count as a fourth face of
regularization. The check is yours.
```

### Validation 5

In [ ]:
validate.check(
    null_dim == 20,
    "the interpolators form a twenty-dimensional affine family",
    "rank 20 in 40 features: every training-consistent model is A+y "
    "plus a null vector — counted over an O(1) gap",
)
validate.check(
    worst_pyth < 1e-10 and all_longer,
    "and A+y is exactly the shortest of them (Eq. 2.4's Pythagoras)",
    f"||w + z||^2 = ||w||^2 + ||z||^2 to {worst_pyth:.0e} on ten draws — "
    "the identity behind implicit regularization, and the reason "
    "minimum-norm interpolation generalizes at all",
)

---
## Notebook summary

**Ridge is a filter, verified as one.** The SVD form matched the
matrix solve at $10^{-15}$ across three $\lambda$; the
$\lambda \to 0$ limit recovered the pseudoinverse at $10^{-11}$;
effective degrees of freedom ran from the rank (15, to $10^{-8}$) to
zero, and both df and the solution norm fell strictly along thirty
points of the path — theorems checked, not curves admired.

**The generalization ledger closed.** Three thousand dedicated-stream
replicates put bias² + variance + $\sigma^2$ within 1% (gate: 3%) of
test error measured on independently noisy targets — a statement
about statistics, since the cross-terms cancel only in expectation —
with variance falling and bias rising across the dial exactly as
{eq}`eq-ll-biasvar` orders.

**The threshold was crossed with the gates watching.** At $p = 40$
the minimum-norm rule interpolated to $10^{-14}$; the test error
spiked near $p = n$ by a factor gated only as *exceeding $10^3\times$
both flanks* (its $10^{16}$ height is the smallest singular value's
business) and descended to 1.0 — which never caught the well-chosen
small model's 0.01, a gate that keeps the notebook from overselling
its own coda. The interpolator family was a 20-dimensional affine
space, and $A^{+}y$ was exactly its shortest member by Pythagoras at
$10^{-15}$.

**Methods introduced.** SVD filter factors, effective degrees of
freedom, dedicated-stream Monte Carlo bias–variance ledgers,
interpolation-threshold sweeps, ordering-gated double descent, and
null-space characterization of interpolators.

## Outlook

- **From closed form to iteration.** [§8.2](gradient-descent-conditioning.ipynb)
  replaces the solve with gradient descent, and Chapter V's
  conditioning returns as the *learning rate's* master constraint —
  the assistant exercise above is its doorway.
- **From one layer to many.** [§8.3](linear-layer-backpropagation.ipynb)
  stacks linear maps with nonlinearities between and differentiates
  through the stack — the chain rule as matrix products, 7.1's
  adjoint strings.
- **The spectrum keeps ruling.** Double descent's spike was a
  singular value; attention's stability ([§8.5](attention-matrix-products.ipynb)),
  LoRA's rank choice ([§8.6](low-rank-lora-quantization.ipynb)) and
  quantization's error floors are all spectral stories — the chapter's
  through-line, inherited from Chapter IV.
- **Honest baselines forever.** The 0.01-versus-1.0 gate — the small
  model winning — is this course's standing posture: measure the
  celebrated phenomenon *and* the boring baseline, and let the gates
  say which one you should ship.

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()